In [ ]:
!pip install -q torch-geometric
!pip install -q networkx
!pip install -q scikit-learn
!pip install -q nltk
!pip install -q pandas numpy

print("Installations complete. You may now run the next cells.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.4 MB/s eta 0:00:00a 0:00:01
Installations complete. You may now run the next cells.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import networkx as nx
import itertools
import io

from torch_geometric.data import Data
from torch_geometric.nn import RGCNConv
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split

import nltk
from nltk.corpus import wordnet as wn
from sklearn.model_selection import train_test_split


# Ensure nltk resources are downloaded
try:
    wn.synsets('dog')
except LookupError:
    nltk.download('wordnet')
    nltk.download('omw-1.4')

from gensim.models import KeyedVectors

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

print("Loading ConceptNet Numberbatch 19.08 from local dataset...")


EMBEDDINGS_PATH = '/kaggle/input/datasets/anuragkacholiya/numberbatch-embedding-english/numberbatch-en.txt'


conceptnet_emb = KeyedVectors.load_word2vec_format(EMBEDDINGS_PATH, binary=False)
EMBEDDING_DIM = 300 

def get_embedding(word):
    """
    Fetches the ConceptNet embedding. 
    The raw 19.08 file uses URIs like '/c/en/word' and replaces spaces with underscores.
    """

    formatted_word = f"/c/en/{word.strip().replace(' ', '_')}"
    
    if formatted_word in conceptnet_emb:
        return conceptnet_emb[formatted_word]
    

    return np.random.normal(scale=0.5, size=(EMBEDDING_DIM,))

print("Embeddings loaded successfully!")

Using device: cuda
Loading ConceptNet Numberbatch 19.08 from local dataset...
Embeddings loaded successfully!


In [ ]:
DATA_PATH = "/kaggle/input/datasets/anuragkacholiya/connections-raw-data/Connections_Data.csv"


df = pd.read_csv(DATA_PATH)


df = df.dropna(subset=["Word"])


df["Word"] = df["Word"].astype(str).str.lower()

puzzles = []
for game_id, group in df.groupby("Game ID"):
    words = group["Word"].tolist()
    

    if len(words) != 16:
        continue
    
    unique_groups = group["Group Name"].unique()
    group_map = {name: idx for idx, name in enumerate(unique_groups)}
    labels = [group_map[name] for name in group["Group Name"]]
    
    puzzles.append({"words": words, "labels": labels})

print(f"Loaded {len(puzzles)+1} puzzles.")



puzzles.append({"words": words, "labels": labels})

print(f"Total clean puzzles: {len(puzzles)}")

train_puzzles, test_puzzles = train_test_split(
    puzzles, test_size=100, random_state=42
)

print(f"Training set: {len(train_puzzles)} puzzles")
print(f"Testing set: {len(test_puzzles)} puzzles")

Loaded 914 puzzles.
Total clean puzzles: 914
Training set: 814 puzzles
Testing set: 100 puzzles


In [ ]:
import json
import networkx as nx
import itertools
from nltk.corpus import wordnet as wn

WIKIDATA_PATH = '/kaggle/input/datasets/anuragkacholiya/wikidata-cache-connections/wikidata_cache.json'
try:
    with open(WIKIDATA_PATH, 'r') as f:
        wikidata_cache = json.load(f)
    print("Successfully loaded local Wikidata cache.")
except FileNotFoundError:
    print("Warning: wikidata_cache.json not found. Tier 2 entity links will be empty.")
    wikidata_cache = {}

def build_tiered_graph(words, tier):
    """Builds a semantic graph using ONLY local data."""
    G = nx.Graph()
    for i, w in enumerate(words):
        G.add_node(w, type='word', target_idx=i)

    # TIER 1: (WordNet)
    if tier >= 1:
        for w1, w2 in itertools.combinations(words, 2):
            syn1 = set(s for s in wn.synsets(w1))
            syn2 = set(s for s in wn.synsets(w2))
            if syn1.intersection(syn2):
                G.add_edge(w1, w2, relation=0)

    # TIER 2: (Offline Wikidata Cache only)
    if tier >= 2:
        for w in words:
            related_entities = wikidata_cache.get(w, [])
            for r in related_entities:
                if r in words and r != w:
                    G.add_edge(w, r, relation=1)

    # TIER 3: Pattern-Based Augmentation (Wordplay)
    if tier >= 3:
        for w1, w2 in itertools.combinations(words, 2):
            if w1[:3] == w2[:3] and len(w1) >= 3: # Prefix match
                G.add_edge(w1, w2, relation=2)
            if len(w1) == len(w2): # Same length match
                G.add_edge(w1, w2, relation=3)

    return G

def prepare_pyg_data(puzzle, tier):
    """Converts NetworkX graph into PyTorch Geometric Data format."""
    words = puzzle["words"]
    true_labels = puzzle["labels"]
    
    G = build_tiered_graph(words, tier)
    node_list = list(G.nodes())
    node_map = {n: i for i, n in enumerate(node_list)}
    
    edges, edge_types = [], []
    for u, v, data in G.edges(data=True):
        edges.append([node_map[u], node_map[v]])
        edges.append([node_map[v], node_map[u]]) # undirected
        edge_types.extend([data["relation"], data["relation"]])
        
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous() if edges else torch.empty((2,0), dtype=torch.long)
    edge_type = torch.tensor(edge_types, dtype=torch.long) if edge_types else torch.empty((0,), dtype=torch.long)
    
    
    x_list = [get_embedding(w) for w in node_list]
    x = torch.tensor(np.array(x_list), dtype=torch.float)
    
    
    pairs = list(itertools.combinations(range(16), 2))
    pair_indices = torch.tensor(pairs, dtype=torch.long).t()
    
    pair_labels = []
    for i, j in pairs:
        pair_labels.append(1.0 if true_labels[i] == true_labels[j] else 0.0)
    pair_labels = torch.tensor(pair_labels, dtype=torch.float)
    
    return x.to(device), edge_index.to(device), edge_type.to(device), pair_indices.to(device), pair_labels.to(device)

Successfully loaded local Wikidata cache.


In [ ]:
class RGCNSolver(nn.Module):
    def __init__(self, in_channels, hidden, num_relations):
        super().__init__()
        self.conv1 = RGCNConv(in_channels, hidden, num_relations)
        self.conv2 = RGCNConv(hidden, hidden, num_relations)
        
        self.scorer = nn.Sequential(
            nn.Linear((in_channels + hidden) * 2, hidden),
            nn.LeakyReLU(0.1), 
            nn.Dropout(0.2),   
            nn.Linear(hidden, 1)
        )

    def forward(self, x, edge_index, edge_type, pair_indices):
        x_orig = x 
        
        x_graph = F.relu(self.conv1(x, edge_index, edge_type))
        x_graph = self.conv2(x_graph, edge_index, edge_type)
        x_combined = torch.cat([x_orig, x_graph], dim=1)
        
        emb_i = x_combined[pair_indices[0]]
        emb_j = x_combined[pair_indices[1]]
        
        diff = torch.abs(emb_i - emb_j)
        
        mult = emb_i * emb_j
        
        
        pair_features = torch.cat([diff, mult], dim=1)
        
        logits = self.scorer(pair_features).squeeze(-1)
        return logits


model = RGCNSolver(in_channels=EMBEDDING_DIM, hidden=64, num_relations=4).to(device)


optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


weight = torch.tensor([4.0]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=weight)

In [ ]:
import os

def train_epoch(puzzles, tier=1):
    model.train()
    total_loss = 0
    for puzzle in puzzles:
        x, edge_index, edge_type, pair_indices, pair_labels = prepare_pyg_data(puzzle, tier)
        
        optimizer.zero_grad()
        logits = model(x, edge_index, edge_type, pair_indices)
        loss = criterion(logits, pair_labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    return total_loss / len(puzzles)


SAVE_PATH = '/kaggle/working/correct_best_rgcn_solver_skipping_newForward.pth'


if os.path.exists(SAVE_PATH):
    print(f"Found saved model at {SAVE_PATH}!")
    print("Skipping training and loading weights directly...")
    

    model.load_state_dict(torch.load(SAVE_PATH, map_location=device))
    

    model.eval() 

else:
    print("No saved model found. Starting training from scratch...")
    best_loss = float('inf')
    
    for epoch in range(50): 
        loss = train_epoch(train_puzzles, tier=3) 
        

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1:02d} | Loss: {loss:.4f}")
            

        if loss < best_loss:
            best_loss = loss
            torch.save(model.state_dict(), SAVE_PATH)
            
    print(f"Training complete! Best model saved to: {SAVE_PATH}")
    

    model.eval()

No saved model found. Starting training from scratch...
Epoch 05 | Loss: 1.0832
Epoch 10 | Loss: 1.0820
Epoch 15 | Loss: 1.0806
Epoch 20 | Loss: 1.0812
Epoch 25 | Loss: 1.0807
Epoch 30 | Loss: 1.0812
Epoch 35 | Loss: 1.0805
Epoch 40 | Loss: 1.0815
Epoch 45 | Loss: 1.0806
Epoch 50 | Loss: 1.0814
Training complete! Best model saved to: /kaggle/working/correct_best_rgcn_solver_skipping_newForward.pth


In [ ]:
loaded_model = RGCNSolver(EMBEDDING_DIM, 64, num_relations=4).to(device)

LOAD_PATH = '/kaggle/working/correct_best_rgcn_solver_newForward.pth'

if os.path.exists(LOAD_PATH):
    loaded_model.load_state_dict(torch.load(LOAD_PATH, map_location=device))

    model = loaded_model 

    model.eval() 
    print("Successfully loaded saved model weights!")
else:
    print("No saved weights found. You are using an untrained model.")

No saved weights found. You are using an untrained model.


In [ ]:
def get_best_partition(pairwise_probs):
    """
    Greedy heuristic to find 4 groups of 4 words maximizing internal validity.
    """
    words_idx = set(range(16))
    groups = []
    

    for _ in range(4):
        best_clique = None
        best_score = -1
        
        for clique in itertools.combinations(words_idx, 4):
            score = sum(pairwise_probs.get((min(i,j), max(i,j)), 0) 
                        for i, j in itertools.combinations(clique, 2))
            if score > best_score:
                best_score = score
                best_clique = clique
                
        groups.append(best_clique)
        words_idx -= set(best_clique)
        

    preds = np.zeros(16, dtype=int)
    for group_idx, group in enumerate(groups):
        for word_idx in group:
            preds[word_idx] = group_idx
            

    total_score = 0
    for group in groups:
        total_score += sum(pairwise_probs.get((min(i,j), max(i,j)), 0) 
                           for i, j in itertools.combinations(group, 2))
    confidence = total_score / 24.0
    return preds, confidence

def solve_progressive(puzzle):
    """Progressive tier activation based on model confidence."""
    model.eval()
    
    for tier in [1, 2, 3]:
        x, edge_index, edge_type, pair_indices, _ = prepare_pyg_data(puzzle, tier)
        
        with torch.no_grad():
            logits = model(x, edge_index, edge_type, pair_indices)
            probs = torch.sigmoid(logits).cpu().numpy()
            
        
        pairs_list = pair_indices.cpu().numpy().T
        pairwise_probs = {(p[0], p[1]): prob for p, prob in zip(pairs_list, probs)}
        
        preds, confidence = get_best_partition(pairwise_probs)
        print(f"  Tier {tier} Confidence: {confidence:.2f}")
        
        
        if confidence > 0.70:
            return preds, tier
            
    return preds, 3


In [ ]:
import itertools
import numpy as np
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

def get_group_overlaps(true_labels, preds):
    """
    Finds the best permutation of labels and returns the number of 
    correctly matched words for each of the 4 individual groups.
    """
    best_overlaps = []
    max_matches = -1
    

    for perm in itertools.permutations([0, 1, 2, 3]):
        mapped_preds = np.array([perm[p] for p in preds])
        overlaps = []
        

        for g in range(4):
            pred_idx = np.where(mapped_preds == g)[0]
            true_idx = np.where(np.array(true_labels) == g)[0]
            overlap = len(set(pred_idx).intersection(set(true_idx)))
            overlaps.append(overlap)
            
        total_matches = sum(overlaps)
        if total_matches > max_matches:
            max_matches = total_matches
            best_overlaps = overlaps
            
    return best_overlaps

print("\nEvaluating Progressive Solver on Unseen Test Set...")

# --- Tracking Metrics ---
total_groups_solved = 0
total_groups_3_correct = 0
games_2_perfect_2_almost = 0
games_solved_completely = 0

total_test_games = len(test_puzzles)

for idx, puzzle in enumerate(test_puzzles):
    
    preds, final_tier = solve_progressive(puzzle)
    true_labels = puzzle["labels"]
    
    ari = adjusted_rand_score(true_labels, preds)
    nmi = normalized_mutual_info_score(true_labels, preds)
    
    overlaps = get_group_overlaps(true_labels, preds)
    
    groups_solved = overlaps.count(4)
    groups_3_correct = overlaps.count(3)
    
    
    total_groups_solved += groups_solved
    total_groups_3_correct += groups_3_correct
    
    
    if groups_solved == 4:
        games_solved_completely += 1
    elif groups_solved == 2 and groups_3_correct == 2:
        games_2_perfect_2_almost += 1

    print(f"\nEvaluating Test Puzzle {idx+1}...")
    print(f"  Stopped at Tier: {final_tier}")
    print(f"  Adjusted Rand Index: {ari:.2f}")
    print(f"  Normalized Mutual Info: {nmi:.2f}")
    print(f"  Group Matches: {overlaps} (Total Words: {sum(overlaps)}/16)")


total_possible_groups = total_test_games * 4

print("\n===== FINAL EVALUATION METRICS =====")
print("1) Grouping Accuracy:")
print(f"   1.1) Total no. of groups solved: {total_groups_solved} (out of {total_possible_groups})")
print(f"   1.2) No. of '3 words correct out of 4 in a group': {total_groups_3_correct}")
print("____")
print(f"2) No. of games in which (2 groups solved completely and remaining 2 groups are '3 words correct'): {games_2_perfect_2_almost}")
print("____")
print(f"3) No. of games solved completely: {games_solved_completely} (out of {total_test_games} test games)")
print("_____")


Evaluating Progressive Solver on Unseen Test Set...
  Tier 1 Confidence: 0.56
  Tier 2 Confidence: 0.55
  Tier 3 Confidence: 0.54

Evaluating Test Puzzle 1...
  Stopped at Tier: 3
  Adjusted Rand Index: -0.15
  Normalized Mutual Info: 0.13
  Group Matches: [1, 2, 1, 2] (Total Words: 6/16)
  Tier 1 Confidence: 0.55
  Tier 2 Confidence: 0.54
  Tier 3 Confidence: 0.54

Evaluating Test Puzzle 2...
  Stopped at Tier: 3
  Adjusted Rand Index: 0.06
  Normalized Mutual Info: 0.34
  Group Matches: [2, 2, 2, 3] (Total Words: 9/16)
  Tier 1 Confidence: 0.59
  Tier 2 Confidence: 0.57
  Tier 3 Confidence: 0.57

Evaluating Test Puzzle 3...
  Stopped at Tier: 3
  Adjusted Rand Index: 0.22
  Normalized Mutual Info: 0.44
  Group Matches: [2, 2, 2, 4] (Total Words: 10/16)
  Tier 1 Confidence: 0.56
  Tier 2 Confidence: 0.56
  Tier 3 Confidence: 0.55

Evaluating Test Puzzle 4...
  Stopped at Tier: 3
  Adjusted Rand Index: 0.43
  Normalized Mutual Info: 0.61
  Group Matches: [3, 3, 2, 4] (Total Words: 12/